# Redes neuronales: evidencia reproducible
UCA · Ciencias de Datos II · Sergio Gevatschnaider

**Datos sintéticos. Ningún valor de este notebook representa una campaña real.**

Prácticas: (1) gradientes verificables, (2) logística frente a MLP para churn, (3) demanda temporal con baseline estacional, Ridge y LSTM CPU.

No necesita API ni GPU. La sección LSTM usa PyTorch, disponible en Colab. Registrar versiones para interpretar diferencias numéricas entre entornos.

[Experiencia del módulo](https://sgevatschnaider.github.io/data-science-business-decisions/modulos/11-redes-neuronales/experiencia/)

In [1]:
import platform, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix,
                             mean_absolute_error, mean_squared_error)
SEED = 42
rng = np.random.default_rng(SEED)
print({'python': platform.python_version(), **{p: importlib.metadata.version(p)
       for p in ['numpy', 'pandas', 'scikit-learn']}, 'seed': SEED})

{'python': '3.12.14', 'numpy': '2.3.5', 'pandas': '2.2.3', 'scikit-learn': '1.8.0', 'seed': 42}


## 1. Forward, backprop y diferencias finitas
Una neurona sigmoide con BCE y logaritmo natural. Se verifica ∂L/∂w=(p−y)x, sin regularización. `logaddexp` evita desbordes al calcular la pérdida desde el logit.

In [2]:
def bce_logit(z, y):
    return np.logaddexp(0, z) - y*z
x, y, w, b = 2., 1., .3, -.1
z = w*x+b
p = 1/(1+np.exp(-z))
gw, gb = (p-y)*x, p-y
eps = 1e-5
fd = (bce_logit((w+eps)*x+b,y)-bce_logit((w-eps)*x+b,y))/(2*eps)
assert np.isclose(gw, fd, atol=1e-8)
w_new, b_new = w-.1*gw, b-.1*gb
pd.DataFrame([{'z':z,'p':p,'BCE':bce_logit(z,y),'dL/dw':gw,'dL/db':gb,
               'diferencia_finita':fd,'w_nuevo':w_new,'b_nuevo':b_new,
               'BCE_nueva':bce_logit(w_new*x+b_new,y)}])

     z         p       BCE     dL/dw     dL/db  diferencia_finita   w_nuevo   b_nuevo  BCE_nueva
0  0.5  0.622459  0.474077 -0.755081 -0.377541          -0.755081  0.375508 -0.062246   0.406926


## 2. Churn sintético: separación y baseline
Simulamos 1.600 observaciones con desbalance. El orden de generación sirve para separar 60/20/20, **sin simular un verdadero proceso temporal ni drift**. En un caso real se definirían fechas de corte, grupos de clientes y disponibilidad de atributos.

Las transformaciones se ajustan dentro de cada pipeline. La validación elige el umbral; test se reserva para la evaluación final. La MLP usa una arquitectura y presupuesto fijados antes de evaluar. No usa test para detenerse.

In [3]:
X, y = make_classification(n_samples=1600, n_features=12, n_informative=7,
    n_redundant=2, weights=[.9,.1], class_sep=.9, random_state=SEED)
Xtr, Xv, Xt = X[:960], X[960:1280], X[1280:]
ytr, yv, yt = y[:960], y[960:1280], y[1280:]
models = {
    'Logística': make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, random_state=SEED)),
    'MLP': make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(16,8),
            alpha=.01, early_stopping=False, n_iter_no_change=20, max_iter=400, tol=1e-5,
            random_state=SEED))}
for model in models.values():
    model.fit(Xtr, ytr)
pd.DataFrame({'split':['train','validación','test'],
              'n':[len(ytr),len(yv),len(yt)],
              'prevalencia':[ytr.mean(),yv.mean(),yt.mean()]})

        split    n  prevalencia
0       train  960     0.100000
1  validación  320     0.121875
2        test  320     0.103125


## 3. Política y utilidad proxy
`U = TP·q·B − contactos·C`. B=100 UM, costo=5 UM y q=0,4 son **supuestos didácticos**. q no se estima a partir del riesgo. El beneficio incremental de una intervención necesita evidencia causal.

AP significa **average precision**, no área trapezoidal de la curva PR. Reportamos ROC-AUC, AP, cantidades de errores y utilidad proxy. Los resultados se calculan al ejecutar, sin una tabla ordinal inventada.

In [4]:
B, C, q = 100., 5., .4
def policy(y_true, score, threshold):
    pred = score >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true,pred,labels=[0,1]).ravel()
    return dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn),
                contactos=int(tp+fp), utilidad_proxy=tp*q*B-(tp+fp)*C)
thresholds = np.linspace(.025,.975,39)
chosen = {}
for name, model in models.items():
    pv = model.predict_proba(Xv)[:,1]
    utilities = [policy(yv,pv,t)['utilidad_proxy'] for t in thresholds]
    chosen[name] = float(thresholds[np.argmax(utilities)])
rows=[]
for name, model in models.items():
    pt=model.predict_proba(Xt)[:,1]
    rows.append({'modelo':name,'umbral_validación':chosen[name],
                 'ROC_AUC_test':roc_auc_score(yt,pt),
                 'AP_test':average_precision_score(yt,pt),
                 **policy(yt,pt,chosen[name])})
churn_results=pd.DataFrame(rows)
churn_results

      modelo  umbral_validación  ROC_AUC_test   AP_test  TP  FP  FN   TN  contactos  utilidad_proxy
0  Logística               0.15      0.854081  0.511908  20  37  13  250         57           515.0
1        MLP               0.10      0.897265  0.630649  24  41   9  246         65           635.0


**Discusión:** ¿un mejor ranking produjo mayor utilidad en el umbral seleccionado? Una única muestra no demuestra superioridad universal. Extensiones: repetir semillas, estimar incertidumbre con unidades independientes y validar calibración. No reajustar el umbral mirando esta tabla test.

## 4. Demanda diaria y ventanas cronológicas
Señal sintética de 420 días con tendencia y ciclo semanal. Cada ventana usa los 28 días anteriores para predecir el siguiente. Train: objetivos anteriores al día 300. Validación: días 300–359. Test: 360–419.

Las ventanas de test contienen pasado observado, que estaría disponible al origen. Es evaluación **de un paso**, no un pronóstico de 60 pasos sin observar nuevas ventas.

In [5]:
t=np.arange(420)
series=100+.08*t+12*np.sin(2*np.pi*t/7)+rng.normal(0,3,len(t))
window=28
origins=np.arange(window,len(series))
WX=np.array([series[j-window:j] for j in origins])
WY=series[origins]
train=origins<300
valid=(origins>=300)&(origins<360)
test=origins>=360
ridge=make_pipeline(StandardScaler(),Ridge(alpha=10.))
ridge.fit(WX[train],WY[train])
seasonal=series[origins[test]-7]
ridge_pred=ridge.predict(WX[test])
def err(name,actual,pred):
    return {'modelo':name,'MAE':mean_absolute_error(actual,pred),
            'RMSE':np.sqrt(mean_squared_error(actual,pred))}
demand_rows=[err('Ingenuo estacional s=7',WY[test],seasonal),
             err('Ridge con 28 rezagos',WY[test],ridge_pred)]
pd.DataFrame(demand_rows)

                   modelo       MAE      RMSE
0  Ingenuo estacional s=7  3.485065  4.363055
1    Ridge con 28 rezagos  2.695531  3.294364


## 5. Walk-forward con ventanas expansivas
Tres orígenes de validación (300, 320 y 340), cada uno con bloque de 14 pronósticos de un paso. El escalado se reajusta usando solo los ejemplos de entrenamiento de cada origen. Las etiquetas finales de test no participan en esta comparación.

In [6]:
fold_rows=[]
for origin in [300,320,340]:
    tr=origins<origin
    va=(origins>=origin)&(origins<origin+14)
    model=make_pipeline(StandardScaler(),Ridge(alpha=10.))
    model.fit(WX[tr],WY[tr])
    fold_rows.append({'origen':origin,**err('Ridge',WY[va],model.predict(WX[va]))})
pd.DataFrame(fold_rows)

   origen modelo       MAE      RMSE
0     300  Ridge  3.147279  3.840801
1     320  Ridge  2.456364  2.888042
2     340  Ridge  3.405466  4.267016


## 6. LSTM pequeña en CPU
PyTorch es necesario para esta celda. No descarga pesos ni datos externos. La escala usa solo valores anteriores al día 300. Se fija una arquitectura pequeña y 30 épocas para la demostración. Validación registra la selección del mejor checkpoint. Test se evalúa después de congelar ese checkpoint.

La semilla reduce variación dentro del mismo entorno; no promete resultados idénticos entre todas las versiones y dispositivos.

In [7]:
RUN_LSTM = True
if RUN_LSTM:
    import torch, copy
    from torch import nn
    torch.manual_seed(SEED)
    torch.set_num_threads(1)
    print('torch',torch.__version__,'device=cpu')
    scale=StandardScaler().fit(series[:300,None])
    tx=torch.tensor(scale.transform(WX.reshape(-1,1)).reshape(-1,window,1),dtype=torch.float32)
    ty=torch.tensor(scale.transform(WY[:,None]),dtype=torch.float32)
    tr_idx=torch.tensor(np.flatnonzero(train))
    va_idx=torch.tensor(np.flatnonzero(valid))
    te_idx=torch.tensor(np.flatnonzero(test))
    class DemandLSTM(nn.Module):
        def __init__(self):
            super().__init__()
            self.rnn=nn.LSTM(input_size=1,hidden_size=8,batch_first=True)
            self.output=nn.Linear(8,1)
        def forward(self,x):
            h,_=self.rnn(x)
            return self.output(h[:,-1,:])
    net=DemandLSTM()
    optim=torch.optim.Adam(net.parameters(),lr=.01)
    criterion=nn.MSELoss()
    best_loss=float('inf')
    training=[]
    for epoch in range(30):
        net.train()
        order=tr_idx[torch.randperm(len(tr_idx))]
        for ids in order.split(32):
            optim.zero_grad()
            loss=criterion(net(tx[ids]),ty[ids])
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(),1.)
            optim.step()
        net.eval()
        with torch.no_grad():
            vl=criterion(net(tx[va_idx]),ty[va_idx]).item()
            tl=criterion(net(tx[tr_idx]),ty[tr_idx]).item()
        training.append((epoch+1,tl,vl))
        if vl<best_loss:
            best_loss=vl
            best=copy.deepcopy(net.state_dict())
    net.load_state_dict(best)
    net.eval()
    with torch.no_grad():
        pred=scale.inverse_transform(net(tx[te_idx]).numpy()).ravel()
    demand_rows.append(err('LSTM 8 unidades',WY[test],pred))
    display(pd.DataFrame(demand_rows))
    curves=pd.DataFrame(training,columns=['época','train_MSE_escalada','valid_MSE_escalada'])
    curves.plot(x='época',ylabel='MSE escalada',title='Curvas medidas sobre señal sintética')
    plt.show()

torch 2.15.0+cpu device=cpu
                   modelo       MAE      RMSE
0  Ingenuo estacional s=7  3.485065  4.363055
1    Ridge con 28 rezagos  2.695531  3.294364
2         LSTM 8 unidades  6.054323  7.252669


## 7. Límites y cierre

**Criterio de éxito:** evidencia reproducible que justifique o rechace complejidad.

### Registro de decisión
Indicar recomendación, evidencia, limitación, responsable y fecha de revisión.

- La muestra de churn es i.i.d. sintética y no simula drift temporal.
- La utilidad usa efectividad supuesta. No mide retención incremental real.
- La demanda es de un paso con pasado observado. Comparar a varios horizontes requiere otro protocolo.
- MAE y RMSE conservan las unidades de demanda. MAPE no es válida cuando y=0 sin una regla explícita.

**Entrega:** baseline, splits, métricas medidas, política congelada, una limitación y condición de rechazo. Si la red pierde frente al baseline, reportarlo.

Fuentes: [Deep Learning, caps. 6–10](https://www.deeplearningbook.org/), [PyTorch LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html), [métricas scikit-learn](https://scikit-learn.org/stable/modules/model_evaluation.html), [Forecasting, cap. 5](https://otexts.com/fpp3/toolbox.html).